In [1]:
import pandas as pd
import ast
import logging
import numpy as np

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset

In [2]:
def preprocess_borg_data(file_path):
    # Load dataset
    df = pd.read_csv(file_path)
    
    # Function to safe-parse dictionary strings (e.g., "{'cpus': 0.02, ...}")
    def parse_dict_column(series, prefix):
        # Convert string representation of dict to actual dict
        dicts = series.apply(lambda x: ast.literal_eval(x) if isinstance(x, str) else x)
        # Flatten into new columns with a prefix
        return pd.json_normalize(dicts).add_prefix(f'{prefix}_')

    # Parse resource requests and actual usage
    req_df = parse_dict_column(df['resource_request'], 'req')
    usage_df = parse_dict_column(df['average_usage'], 'usage')

    # Join back to main dataframe and drop original complex columns
    df = pd.concat([df.drop(['resource_request', 'average_usage'], axis=1), req_df, usage_df], axis=1)

    # Feature Engineering: Resource Efficiency
    # Tasks nearing 1.0 (100% usage) are often high-risk for failure
    df['cpu_efficiency'] = df['usage_cpus'] / (df['req_cpus'] + 1e-9)
    df['mem_efficiency'] = df['usage_memory'] / (df['req_memory'] + 1e-9)
    
    # Drop high-cardinality/anonymized strings for simple modeling
    cols_to_drop = ['user', 'collection_name', 'collection_logical_name', 'machine_id']
    df = df.drop(columns=[c for c in cols_to_drop if c in df.columns])

    # Handle missing values
    df = df.fillna(0)
    
    return df

In [3]:
def setup_logger(file_name="training.log"):
    # Create a logger
    logger = logging.getLogger("train_logger")
    logger.setLevel(logging.INFO)
    
    # Create file handler which logs even debug messages
    fh = logging.FileHandler(file_name)
    fh.setLevel(logging.INFO)
    
    # Create formatter and add it to the handler
    formatter = logging.Formatter('%(asctime)s - %(levelname)s - %(message)s')
    fh.setFormatter(formatter)
    
    # Add the handler to the logger
    logger.addHandler(fh)
    return logger

In [4]:
processed_df = preprocess_borg_data('borg_traces_data.csv')

In [5]:
class FailurePredictorLSTM(nn.Module):
    def __init__(self, input_size, hidden_size=64):
        super(FailurePredictorLSTM, self).__init__()
        self.hidden_size = hidden_size
        # The GRU keeps the 'internal state' of the task
        self.gru = nn.LSTM(input_size, hidden_size, batch_first=True)
        self.classifier = nn.Linear(hidden_size, 1)
        self.sigmoid = nn.Sigmoid()

    def forward(self, x, hidden=None):
        # x: [batch, seq_len, features]
        # out: [batch, seq_len, hidden_size]
        out, hidden = self.gru(x, hidden)
        
        # Predict failure probability for the latest time step
        prediction = self.sigmoid(self.classifier(out[:, -1, :]))
        return prediction, hidden

# Initialize model
# Features: priority, scheduling_class, req_cpus, req_mem, usage_cpus, usage_mem (6 total)
model = FailurePredictorLSTM(input_size=6)


In [6]:
# Hyperparameters
input_size = 6   # Our 6 numeric features
hidden_size = 128
num_epochs = 300

# Initialize model, loss, and optimizer
model = FailurePredictorLSTM(input_size, hidden_size)
criterion = nn.BCELoss() # Binary Cross Entropy for failure (0 or 1)
optimizer = optim.Adam(model.parameters(), lr=0.001)

def train_with_detailed_metrics(model, loader, logger, epochs=num_epochs, threshold=0.5):
    optimizer = optim.Adam(model.parameters(), lr=0.001)
    criterion = nn.BCELoss() # Binary Cross Entropy

    header = f"{'Epoch':<8} | {'Loss':<7} | {'Acc':<7} | {'Prec':<7} | {'FPR':<7} | {'FNR':<7}"
    separator = "-" * 65

    logger.info(header)
    logger.info(separator)

    for epoch in range(epochs):
        model.train()
        all_preds = []
        all_targets = []
        epoch_loss = 0

        for batch_X, batch_y in loader:
            optimizer.zero_grad()
            
            # Forward pass
            probs, _ = model(batch_X)
            probs = probs.squeeze()
            
            loss = criterion(probs, batch_y)
            loss.backward()
            optimizer.step()
            
            epoch_loss += loss.item()

            # Store predictions and targets for metrics
            # We use a threshold (default 0.5) to convert probabilities to 0 or 1
            preds = (probs > threshold).float()
            all_preds.extend(preds.detach().cpu().numpy())
            all_targets.extend(batch_y.detach().cpu().numpy())

        # Calculate Final Metrics for the Epoch
        all_preds = np.array(all_preds)
        all_targets = np.array(all_targets)

        tp = np.sum((all_preds == 1) & (all_targets == 1))
        tn = np.sum((all_preds == 0) & (all_targets == 0))
        fp = np.sum((all_preds == 1) & (all_targets == 0))
        fn = np.sum((all_preds == 0) & (all_targets == 1))

        # Metrics Formulas
        accuracy = (tp + tn) / len(all_targets)
        precision = tp / (tp + fp) if (tp + fp) > 0 else 0
        
        # Error Rates
        fpr = fp / (fp + tn) if (fp + tn) > 0 else 0 # False Positive Rate (Type I)
        fnr = fn / (fn + tp) if (fn + tp) > 0 else 0 # False Negative Rate (Type II)

        # Print formatted results
        metrics_str = f"{epoch+1:<8} | {epoch_loss/len(loader):<7.4f} | {accuracy:<7.2%} | {precision:<7.2f} | {fpr:<7.2f} | {fnr:<7.2f}"
        print(metrics_str)
        logger.info(metrics_str)

In [7]:
processed_df

,Unnamed: 0,time,instance_events_type,collection_id,scheduling_class,collection_type,priority,alloc_collection_id,instance_index,constraint,...,tail_cpu_usage_distribution,cluster,event,failed,req_cpus,req_memory,usage_cpus,usage_memory,cpu_efficiency,mem_efficiency
0,0,0,2,94591244395,3,1,200,0,144,[],...,[0.00535583 0.00541687 0.00548553 0.00554657 0...,7,FAIL,1,0.020660,0.014435,0.004662,5.920410e-03,0.225628,0.410148
1,1,2517305308183,2,260697606809,2,0,360,221495397286,335,[],...,[1.23977661e-05 1.23977661e-05 1.23977661e-05 ...,7,FAIL,1,0.007240,0.001303,0.000000,9.536743e-07,0.000000,0.000732
2,2,195684022913,6,276227177776,2,0,103,0,376,[],...,[0.02902222 0.02929688 0.0295105 0.0296936 0...,7,SCHEDULE,0,0.048584,0.004166,0.024200,2.788544e-03,0.498116,0.669414
3,3,0,2,10507389885,3,0,200,0,1977,[],...,[0.05535889 0.05584717 0.05633545 0.05718994 0...,8,FAIL,1,0.070435,0.041626,0.047607,3.442383e-02,0.675910,0.826979
4,4,1810627494172,3,25911621841,2,0,0,0,3907,[],...,[0.00041485 0.00041485 0.00041485 0.00041485 0...,2,FINISH,0,0.002449,0.000232,0.000271,7.629395e-05,0.110592,0.329217
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
405889,405889,0,2,299950285958,1,0,117,0,1341,[],...,[0.0222168 0.02261353 0.02294922 0.02404785 0...,8,FAIL,1,0.035950,0.015488,0.015320,1.501083e-03,0.426146,0.096921
405890,405890,18279167039,0,330587213089,2,0,360,330587120885,898,[],...,[0.05828857 0.05871582 0.05938721 0.06097412 0...,1,ENABLE,0,0.021973,0.001303,0.038147,1.998901e-02,1.736111,15.344058
405891,405891,733307641549,3,13748637767,3,0,200,13748635759,1959,[{'name': 'GKAYWlOFlntxaxFt+CCHj/Og1BgToNx62SM...,...,[0.03515625 0.03552246 0.03729248 0.03912354 0...,2,FINISH,0,0.121460,0.082642,0.023560,7.580566e-02,0.193970,0.917282
405892,405892,1984523129238,2,323372663325,1,0,117,0,6452,[{'name': '5dEuieuWMFy+CNMBBf/uXNX5nP4Kgzeu0O6...,...,[0.00904846 0.00907898 0.00909424 0.00912476 0...,8,FAIL,1,0.005669,0.001562,0.006004,9.822845e-04,1.059219,0.628815


In [8]:
def create_sliding_windows(df, window_size=5):
    X, y = [], []
    # Ensure tasks don't bleed into each other
    for cid, group in df.groupby('collection_id'):
        # Sort by time if your 'time' column isn't already sorted
        data = group[['priority', 'scheduling_class', 'req_cpus', 'req_memory', 'usage_cpus', 'usage_memory']].values
        targets = group['failed'].values
        
        # Create windows: use rows [i : i+5] to predict row [i+5]
        for i in range(len(data) - window_size):
            X.append(data[i : i + window_size])
            y.append(targets[i + window_size])
            
    return torch.tensor(np.array(X)).float(), torch.tensor(np.array(y)).float()

# Example usage with your preprocessed dataframe
X_train, y_train = create_sliding_windows(processed_df)
dataset = TensorDataset(X_train, y_train)
train_loader = DataLoader(dataset, batch_size=32, shuffle=True)

In [9]:
train_logger = setup_logger("lstm.log")
train_with_detailed_metrics(model, train_loader, logger=train_logger)

1        | 0.4829  | 77.68%  | 0.59    | 0.00    | 0.98   
2        | 0.4748  | 77.76%  | 0.57    | 0.01    | 0.95   
3        | 0.4719  | 77.83%  | 0.58    | 0.01    | 0.95   
4        | 0.4697  | 77.83%  | 0.58    | 0.01    | 0.95   
5        | 0.4680  | 77.86%  | 0.59    | 0.01    | 0.95   
6        | 0.4663  | 77.89%  | 0.58    | 0.01    | 0.94   
7        | 0.4643  | 77.94%  | 0.60    | 0.01    | 0.94   
8        | 0.4620  | 77.95%  | 0.60    | 0.01    | 0.94   
9        | 0.4608  | 78.02%  | 0.61    | 0.01    | 0.93   
10       | 0.4593  | 78.03%  | 0.60    | 0.01    | 0.93   
11       | 0.4566  | 78.07%  | 0.60    | 0.02    | 0.92   
12       | 0.4555  | 78.11%  | 0.59    | 0.02    | 0.91   
13       | 0.4541  | 78.18%  | 0.60    | 0.02    | 0.91   
14       | 0.4537  | 78.19%  | 0.59    | 0.02    | 0.90   
15       | 0.4553  | 78.15%  | 0.59    | 0.02    | 0.91   
16       | 0.4535  | 78.23%  | 0.60    | 0.02    | 0.90   
17       | 0.4518  | 78.33%  | 0.60    | 0.02    | 0.89 

In [10]:
checkpoint = {
    'model_state_dict': model.state_dict(),
    'optimizer_state_dict': optimizer.state_dict(),
}
torch.save(checkpoint, 'lstm.pth')

In [11]:
checkpoint = torch.load('lstm.pth')
model.load_state_dict(checkpoint['model_state_dict'])
optimizer.load_state_dict(checkpoint['optimizer_state_dict'])

In [13]:
train_with_detailed_metrics(model, train_loader, logger=train_logger, epochs=1)

1        | 0.3585  | 82.69%  | 0.72    | 0.04    | 0.63   
